# 09_03 The search as an MCP tool, for Claude and other apps

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → [MCP tool]
> ```
>
> You are here: **after the main course**: a desk where any AI app can ask the librarian
>
> Project files this connects to:
>
> - `src/homer_ai/homer_mcp_server.py`
> - `src/homer_ai/retrieval.py`
> - `tests/test_mcp_server.py`
>
> **This notebook:** The search runs in a notebook. How do Claude and other AI apps get to call it, and what must they be told about its limits?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How the search becomes a second tool, `search_homer`, on the server of 06_03
- What the tool's description has to tell the model, including what the search can't do
- How to talk to the server from Python, the way an AI app does
- How to connect the server to Claude Code, Claude Desktop, Codex CLI, LM Studio and nanobot

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **lazy loading** | doing a slow piece of work only when it's first needed, then keeping the result | the search index is built by the first search |

Words you've met before: **MCP client** ([06_00](../06_making_it_talk/06_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ The search works in a notebook. How does Claude get to use it?

The same way it got to use our model in [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb): through the MCP server. A tool there is a Python function with `@mcp.tool()` above it, type hints for its arguments, and a docstring that the model reads as the description.

So we add one function to `homer_mcp_server.py`. The server then offers two tools:

| Tool | What it gives | Is the text real? |
|---|---|---|
| `consult_homer_llm` | our model's continuation of a Greek seed | no, it only looks like Greek |
| `search_homer` | passages of the Odyssey and the Iliad, with `poem book:line` | yes, copied from `data/homer/` |

The search tool doesn't answer anything itself. It fetches. The app's model reads the passages and writes the answer, and because every passage starts with its reference, it can cite. In the picture of [09_00](09_00_chapter_overview.ipynb), this is the desk where any student can ask the librarian.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what does the model see of the new tool?

Its name, its arguments and its description. Let's ask the server, without starting it.

```
search_homer(question, poem='both', language='english', k=5)
```

In [ ]:
from homer_ai.lessons import import_mcp_server

server = import_mcp_server()
tools = {tool.name: tool for tool in await server.mcp.list_tools()}
print("tools:", list(tools))
print()
tool = tools["search_homer"]
for name, details in tool.input_schema["properties"].items():
    print(f"{name:9} {details}")
print("must give:", tool.input_schema["required"])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two tools now. `search_homer` has four arguments, and only `question` is required.

Look at `poem` and `language`. In the code their type is `Literal["odyssey", "iliad", "both"]`, and that became an `enum` in what the model sees: a list of the only values allowed. The model can't invent a third poem, and if it tries, the call is refused before our function runs.

</details>

### ❓ Step 2: what does the description have to say?

Everything the model needs to use the tool well, and that includes what it can't do. A model that thinks the search understands meaning will ask it vague questions and trust weak results.

👉 **Predict first:** from what you saw in 09_02, what should the description warn about?

In [ ]:
print(tool.description)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three warnings, each from something we measured or saw:

- **Roman names.** Butler says Ulysses, Jove, Minerva. The model should expect them in the passages.
- **Line numbers are ours.** They're lines of this project's files. A scholar's *Odyssey 9.366* is a verse number, which is something else, and the model shouldn't mix them up.
- **Words, not meaning.** The search misses a passage that says it in other words, so the model is told to try again with words the text would use, and to say so when the passages don't hold the answer.

That last line matters most. Without it, a model handed five passages tends to squeeze an answer out of them.

</details>

### ❓ Step 3: what does a call return?

Plain text: each passage with its reference in square brackets and its score. We can call the function directly.

👉 **Predict first:** we ask for `k=500` passages. How many come back?

In [ ]:
result = server.search_homer("Who recognises Odysseus by the scar on his leg?", k=2)
print(result[:420] + "…")
print()
many = server.search_homer("Cyclops", k=500)
print("asked for 500, got", many.count("] (score "))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first passage is `[odyssey 19:25]`, with a score of 15.7. It's the paragraph where the old nurse is about to wash his feet, and he remembers the scar. (We print only the start. The call returned two passages.)

Asked for 500, we got 10. `k` is clamped to 1 to 10, like `chars_to_generate` in 06_03, so one call can't flood the model's context with text. A passage longer than 1,200 characters is cut there and ends in `…`.

The lines that begin `homer: search …` are the server's log. Notebooks show them in red or in a separate box. It goes to stderr, because stdout belongs to the MCP messages.

</details>

### ❓ Step 4: does the server start more slowly now?

No, because the index is built by the first search and not at startup. Doing slow work only when it's first needed is called **lazy loading**.

It matters for a server like ours. The client starts it for every session, often to call the other tool or none at all. Building the index for both poems in English takes about a quarter of a second, and it's kept in memory after that.

In [ ]:
import time

from homer_ai import retrieval

retrieval._indexes.clear()  # forget the indexes, as in a server that has just started
for attempt in ("first call", "second call"):
    started = time.perf_counter()
    server.search_homer("Sirens wax", poem="odyssey", k=1)
    print(f"{attempt}: {(time.perf_counter() - started) * 1000:.0f} ms")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first call takes around a tenth of a second, because it reads the 24 files of the Odyssey and builds their index. The second takes less than a millisecond.

</details>

### ❓ Step 5: can I talk to the server the way Claude does?

Yes. The `mcp` library has a client as well. The next cell starts our server as a separate program, connects to it over stdio (its standard input and output, see [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb)), asks for the tool list and calls the search. That's what every **MCP client** does, whether it's Claude Code or another app.

In [ ]:
import os
import sys

from mcp import ClientSession, StdioServerParameters, stdio_client

params = StdioServerParameters(
    command=sys.executable,  # the Python this notebook runs on
    args=["-m", "homer_ai.homer_mcp_server"],  # what `uv run homer-serve` runs
    env=dict(os.environ),  # the notebook's settings, e.g. $HOMER_MODEL
)
async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        print("tools:", [tool.name for tool in listed.tools])
        arguments = {"question": "Sirens wax", "poem": "odyssey", "k": 1}
        reply = await session.call_tool("search_homer", arguments)
        print(reply.content[0].text[:150] + "…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A real server process answered: two tools, and for the search a passage from Book 12 of the Odyssey, the Sirens. Nothing in this exchange is special to Claude. Any program that speaks MCP can do the same.

</details>

### ❓ Step 6: how do I connect it to my AI app?

Every client needs the same three facts: a name for the server, the command that starts it, and the command's arguments. For us that's

```
name:      homer
command:   uv
arguments: --directory /path/to/homer-ai run homer-serve
```

`--directory` lets `uv` find the project from wherever the app happens to start. The clients only differ in where you write this down. The next cell prints it for five of them, with your own project path filled in.

In [ ]:
import json

from homer_ai.retrieval import HOMER

project = str(HOMER.parents[1])  # the folder that holds pyproject.toml
arguments = ["--directory", project, "run", "homer-serve"]
entry = {"command": "uv", "args": arguments}

print("Claude Code (run in a terminal):")
print("  claude mcp add homer -- uv", " ".join(arguments))
print()
print("Claude Desktop (claude_desktop_config.json) and LM Studio (mcp.json):")
print(json.dumps({"mcpServers": {"homer": entry}}, indent=2))
print()
print("Codex CLI (~/.codex/config.toml):")
print(f'[mcp_servers.homer]\ncommand = "uv"\nargs = {json.dumps(arguments)}')
print()
print("nanobot, the HKUDS project (~/.nanobot/config.json):")
print(json.dumps({"tools": {"mcpServers": {"homer": entry}}}, indent=2))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Four spellings of the same three facts. JSON with `mcpServers` for Claude Desktop and LM Studio, a TOML table named `mcp_servers.homer` for Codex, and the same JSON one level deeper, under `tools`, for nanobot.

</details>

### ❓ Where does each client keep its settings, and how do I check it worked?

I read each client's own documentation on 24 November 2026. These tools change quickly, so the last column says how sure I am. If something differs on your machine, the linked page wins.

| Client | Where it goes | How to check | Checked against the docs? |
|---|---|---|---|
| **Claude Code** | the command `claude mcp add homer -- uv --directory … run homer-serve` (add `--scope user` before the name for all your projects; `--scope project` writes a shareable `.mcp.json` with an `mcpServers` entry) | `claude mcp list`, or `/mcp` inside a session | yes: [code.claude.com/docs/en/mcp](https://code.claude.com/docs/en/mcp). This is the one I ran myself, with the `mcpServers` entry. |
| **Claude Desktop** | Settings → Developer → Edit Config opens `claude_desktop_config.json` (macOS: `~/Library/Application Support/Claude/`, Windows: `%APPDATA%\Claude\`). Quit the app fully and start it again. | the server and its tools appear under the connectors menu of the message box | yes: [modelcontextprotocol.io/docs/develop/connect-local-servers](https://modelcontextprotocol.io/docs/develop/connect-local-servers). Not run by me. |
| **Codex CLI** | `~/.codex/config.toml`, or the command `codex mcp add homer -- uv --directory … run homer-serve` | `codex mcp list`, or `/mcp` inside Codex | yes: OpenAI's Codex MCP page (it has moved between sites; search for *Codex MCP*). Not run by me. |
| **LM Studio** | in the app: the Program tab in the right sidebar → Install → Edit mcp.json | the tools show up in the chat's tool list | partly: [lmstudio.ai/docs/app/mcp](https://lmstudio.ai/docs/app/mcp) says the file follows Cursor's `mcp.json` notation with `mcpServers`, but its only example is a remote server. The `command` and `args` form above is taken from that notation. **Unverified** for LM Studio itself. |
| **nanobot** (HKUDS) | `~/.nanobot/config.json`, under `tools` → `mcpServers` | start `nanobot gateway --verbose` and read the startup log | partly: the project's `docs/configuration.md` shows `command` and `args` there, with `npx` examples. Using `uv` as the command is **unverified**. |

Two things trip people up with any of them:

- **The path must be absolute.** `~` and relative paths are often not expanded.
- **Desktop apps may not find `uv`.** An app started from the dock doesn't always have your terminal's `PATH`. If the server doesn't start, put the full path of `uv` as the command (`which uv` in a terminal prints it; on Windows, `where uv`).

There's a second, unrelated project called nanobot (`nanobot-ai/nanobot`, written in Go). Its README only shows servers reached by URL, so I give no snippet for it.

### ❓ And then? What do I type?

A question, in plain words:

> Using the homer tools: what does the ghost of Achilles say about being dead? Quote the passage and give its reference.

The model reads the two tool descriptions, picks `search_homer`, gets passages back, and answers from them. Ask the same thing without *using the homer tools* and it may answer from memory. That's its choice, and often a reasonable one. [09_04](09_04_with_and_without.ipynb) is about what the difference is worth.

You can bring the other tool in too:

> …and then let the Homer model continue the Greek word μῆνιν for 40 letters.

That calls `consult_homer_llm`. What comes back looks like Greek and mostly isn't, and its description says so, so a careful model will tell you.

*A tool description is the cheapest thing you'll ever tune. When a model misuses one of my tools, I rewrite the description before I touch any code.*

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why does the tool return text and not JSON?

Because its reader is a language model, and the references are what it has to copy. A line like `[odyssey 10:24]` right above the passage is hard to get wrong. JSON would work too, and is the better choice when a program reads the result.

### ❓ Does the language have to be English?

No. With `language="greek"` the tool searches the original, in passages of 5 verses, and accents may be left out: `μηνιν αειδε` finds `iliad 1:1-5`. But the question must then be Greek words. BM25 matches words, so an English question finds nothing in Greek text. A search across languages needs a model that maps both into the same vectors.

### ❓ Is it safe to let an AI app run this server?

It only reads files in `data/homer/` and the model file, writes nothing, and makes no network calls. The inputs are bounded: `poem` and `language` take only the listed values, `k` is clamped, and long passages are cut. `tests/test_mcp_server.py` checks each of those.

## 5. Recap

- `search_homer` is a second tool on the same MCP server: a typed function with an honest docstring.
- Its description tells the model about Roman names, about our own line numbers, and that the search matches words and can miss.
- `k` is clamped to 1 to 10, the allowed poems and languages are an `enum`, and the index is built by the first search, so the server starts fast.
- Every client needs a name, a command and its arguments. Only the file and the spelling differ.

**Next:** [09_04: With and without the passages, and where this goes in real work](09_04_with_and_without.ipynb).